# E18 异常检测：IsolationForest（无标签场景）

> 覆盖矩阵：无监督异常 · 区别于 E05 有标签欺诈

## 你将学到
1. 无标签时用隔离森林 / LOF；评估需注入已知异常或业务抽检。
2. 分数阈值与告警预算。
3. **缺口**：无 `AnomalyDetector` 封装。


In [ ]:
QUICK = True
RANDOM_STATE = 42
from wanglaoshi import ready
ready(seed=RANDOM_STATE)


## 1. 合成正常 + 注入异常

In [ ]:
import numpy as np
import pandas as pd
from wanglaoshi import DataAnalyzer

rng = np.random.default_rng(RANDOM_STATE)
n_norm, n_out = 500, 25
X_norm = rng.normal(0, 1, size=(n_norm, 4))
X_out = rng.normal(4, 1.2, size=(n_out, 4))
X = np.vstack([X_norm, X_out])
y_true = np.array([0] * n_norm + [1] * n_out)  # 仅用于事后评估
df = pd.DataFrame(X, columns=[f'f{i}' for i in range(4)])
df['is_injected_outlier'] = y_true
print(DataAnalyzer(df.drop(columns=['is_injected_outlier'])).get_basic_info().get('shape'))


## 2. IsolationForest

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_fscore_support
from pathlib import Path

clf = IsolationForest(contamination=n_out / len(df), random_state=RANDOM_STATE)
pred = clf.fit_predict(df[[f'f{i}' for i in range(4)]])  # -1 异常
y_hat = (pred == -1).astype(int)
p, r, f1, _ = precision_recall_fscore_support(y_true, y_hat, average='binary', zero_division=0)
print(dict(precision=p, recall=r, f1=f1, flagged=int(y_hat.sum())))
out = Path('outputs/18_anomaly_detection'); out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({'precision': [p], 'recall': [r], 'f1': [f1]}).to_csv(out / 'anomaly_metrics.csv', index=False)


## 3. 与有监督欺诈课的边界

In [ ]:
print('E05 Fraud：有标签 → PR-AUC / class_weight')
print('E18：无标签 → 污染率假设 + 人工复核；分数需业务校准')


## 附录
- 作业：改 contamination，画 precision-recall 随阈值变化。
- 包优化：`AnomalyDetector.fit_score(df)` → 分数、阈值建议、规则分融合。
